In [1]:
import os
# disable oneDNN optimizations and silences most TF log messages
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL']   = '2'

import random
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shutil

# Import sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Import keras helpers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.losses import CategoricalCrossentropy

2025-04-19 12:22:36.321146: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745079756.340310  279535 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745079756.346083  279535 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745079756.360811  279535 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745079756.360829  279535 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745079756.360831  279535 computation_placer.cc:177] computation placer alr

In [7]:
PRE_TRAIN_WIDTH    = 224
PRE_TRAIN_HEIGHT   = 224
NUM_CLASSES        = 10
HEAD_UNITS_1       = 256
HEAD_UNITS_2       = 128
L2_WEIGHT_DECAY    = 1e-5
DROPOUT_RATE       = 0.3
INITIAL_LR         = 1e-3
LABEL_SMOOTHING    = 0.05
CLASS_NAMES        = [str(i) for i in range(NUM_CLASSES)]
IMAGE_WIDTH   = 28
IMAGE_HEIGHT  = 28
IMAGE_DEPTH   = 1
BATCH_SIZE    = 64
NUM_CLASSES   = 10
EPOCHS = 30

In [10]:
# set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [11]:
DATA_RAW_DIR       = "data/raw"
DATA_PROCESSED_DIR = "data/processed"
FIGURES_DIR        = "outputs/figures"
MODELS_DIR         = "outputs/models"
RESULTS_DIR        = "results"

In [12]:
# load processed splits
train_npz = np.load(os.path.join(DATA_PROCESSED_DIR, "train.npz"))
val_npz   = np.load(os.path.join(DATA_PROCESSED_DIR, "val.npz"))
test_npz  = np.load(os.path.join(DATA_PROCESSED_DIR, "test.npz"))

x_train, y_train = train_npz["x"], train_npz["y"]
x_val,   y_val   = val_npz["x"],   val_npz["y"]
x_test,  y_test  = test_npz["x"],  test_npz["y"]

# one‑hot encode labels
y_train = to_categorical(y_train, NUM_CLASSES)
y_val   = to_categorical(y_val,   NUM_CLASSES)
y_test  = to_categorical(y_test,  NUM_CLASSES)

# create image data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# build generators
train_generator = train_datagen.flow(
    x_train.reshape(-1, IMAGE_WIDTH, IMAGE_HEIGHT, IMAGE_DEPTH),
    y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)
validation_generator = val_datagen.flow(
    x_val.reshape(-1, IMAGE_WIDTH, IMAGE_HEIGHT, IMAGE_DEPTH),
    y_val,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_generator = test_datagen.flow(
    x_test.reshape(-1, IMAGE_WIDTH, IMAGE_HEIGHT, IMAGE_DEPTH),
    y_test,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# 1) extended OAT grids
lr_values         = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
dropout_values    = [0.1, 0.2, 0.3, 0.5, 0.5]
wd_values         = [1e-1,1e-2,1e-3, 1e-4, 1e-5]
ls_values         = [0.0, 0.05, 0.1, 0.15, 0.2]
sched_styles      = ['none', 'plateau', 'cosine']

results = []                     

def cosine_schedule(epoch, lr):
    return lr * 0.5 * (1 + np.cos(np.pi * epoch / max(EPOCHS,1)))

# ---------- build a model with given h‑params ----------
def build_sweep_model(lr, dropout, wd, ls, sched_style):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(PRE_TRAIN_HEIGHT, PRE_TRAIN_WIDTH, 3),
        include_top=False, weights="imagenet"
    )
    base.trainable = False

    inp = tf.keras.Input((IMAGE_WIDTH, IMAGE_HEIGHT, IMAGE_DEPTH))
    x   = tf.keras.layers.Resizing(PRE_TRAIN_HEIGHT, PRE_TRAIN_WIDTH)(inp)
    x   = tf.keras.layers.Concatenate()([x, x, x])
    x   = base(x, training=False)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)

    for units in (HEAD_UNITS_1, HEAD_UNITS_2):
        x = tf.keras.layers.Dense(units,
                kernel_regularizer=regularizers.l2(wd), use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.Dropout(dropout)(x)

    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=CategoricalCrossentropy(label_smoothing=ls),
        metrics=["accuracy"]
    )

    if sched_style == 'plateau':
        cb = ReduceLROnPlateau(monitor='val_accuracy',
                               factor=0.5, patience=2, min_lr=1e-7)
    elif sched_style == 'cosine':
        cb = LearningRateScheduler(cosine_schedule, verbose=0)
    else:
        cb = None
    return model, cb

# ---------------- one‑epoch OAT sweeps -----------------
def run_one(param_name, value, **kwargs):
    m, cb = build_sweep_model(**kwargs)
    h = m.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=1, callbacks=[cb] if cb else [],
        verbose=0
    )
    acc = h.history['val_accuracy'][-1]
    results.append({'param': param_name, 'value': value,
                    'val_accuracy': round(float(acc), 6)})

# learning rate
for lr in lr_values:
    run_one('learning_rate', lr,
            lr=lr, dropout=DROPOUT_RATE, wd=L2_WEIGHT_DECAY,
            ls=LABEL_SMOOTHING, sched_style='none')

# dropout
for d in dropout_values:
    run_one('dropout_rate', d,
            lr=INITIAL_LR, dropout=d, wd=L2_WEIGHT_DECAY,
            ls=LABEL_SMOOTHING, sched_style='none')

# weight decay
for wd in wd_values:
    run_one('weight_decay', wd,
            lr=INITIAL_LR, dropout=DROPOUT_RATE, wd=wd,
            ls=LABEL_SMOOTHING, sched_style='none')

# label smoothing
for ls in ls_values:
    run_one('label_smoothing', ls,
            lr=INITIAL_LR, dropout=DROPOUT_RATE, wd=L2_WEIGHT_DECAY,
            ls=ls, sched_style='none')

# scheduler style
for style in sched_styles:
    run_one('scheduler_style', style,
            lr=INITIAL_LR, dropout=DROPOUT_RATE, wd=L2_WEIGHT_DECAY,
            ls=LABEL_SMOOTHING, sched_style=style)

# ---------------- save & show table -------------------
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

csv_path = os.path.join(RESULTS_DIR, "oat_smoke_results.csv")
df_prev  = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
df_full  = pd.concat([df_prev, pd.DataFrame(results)], ignore_index=True).drop_duplicates()
df_full.to_csv(csv_path, index=False)
print("\n=== OAT SMOKE RESULTS ===")
print(df_full)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

numeric_params = {"learning_rate", "dropout_rate", "weight_decay", "label_smoothing"}

for param in df_full["param"].unique():
    sub = df_full[df_full["param"] == param].copy()

    # cast to float if this parameter is numeric
    if param in numeric_params:
        sub["value"] = sub["value"].astype(float)
        sub = sub.sort_values("value")           # nicer left→right order

    plt.figure(figsize=(6,4))
    if param == "learning_rate":
        plt.semilogx(sub["value"], sub["val_accuracy"], marker="o")
    else:
        plt.plot(sub["value"], sub["val_accuracy"], marker="o")

    plt.title(f"smoke test: {param}")
    plt.xlabel(param)
    plt.ylabel("val_accuracy")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIGURES_DIR, f"sweep_{param}.jpg"),
        dpi=300,
        format="jpg"
    )
    plt.show()

In [ ]:
import os, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt

lr_grid   = [5e-4, 1e-3, 2e-3]
ls_grid   = [0.05, 0.10, 0.15]
do_grid   = [0.1, 0.2, 0.3]
wd        = 1e-5                      
patience  = 3                         
max_epochs= 10

grid_results = []

def build_grid_model(lr, dropout, ls):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(PRE_TRAIN_HEIGHT, PRE_TRAIN_WIDTH, 3),
        include_top=False, weights="imagenet"
    )
    base.trainable = False

    inp = tf.keras.Input((IMAGE_WIDTH, IMAGE_HEIGHT, IMAGE_DEPTH))
    x   = tf.keras.layers.Resizing(PRE_TRAIN_HEIGHT, PRE_TRAIN_WIDTH)(inp)
    x   = tf.keras.layers.Concatenate()([x, x, x])
    x   = base(x, training=False)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)

    for units in (HEAD_UNITS_1, HEAD_UNITS_2):
        x = tf.keras.layers.Dense(units,
                kernel_regularizer=regularizers.l2(wd), use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.Dropout(dropout)(x)

    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss=CategoricalCrossentropy(label_smoothing=ls),
        metrics=["accuracy"]
    )
    return model

# run grid
for lr, ls, do in itertools.product(lr_grid, ls_grid, do_grid):
    print(f"lr={lr}, ls={ls}, dropout={do}")
    m = build_grid_model(lr, do, ls)
    es = EarlyStopping(monitor='val_accuracy',
                       patience=patience, restore_best_weights=True, verbose=0)
    h = m.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=max_epochs,
        callbacks=[es],
        verbose=0
    )
    best_acc = max(h.history['val_accuracy'])
    grid_results.append({'lr': lr, 'label_smooth': ls,
                         'dropout': do, 'val_accuracy': round(float(best_acc),6)})

# DataFrame & save
df_grid = pd.DataFrame(grid_results)
os.makedirs("results", exist_ok=True)
df_grid.to_csv("results/grid_results.csv", index=False)
print(df_grid.sort_values("val_accuracy", ascending=False))